Run this first in a seperate terminal: `python -m mlflow server --host 127.0.0.1 --port 5000`

# Import libraries

In [16]:
import mlflow
import mlflow.sklearn
import pandas as pd
import psycopg2
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

# Set DB configuration

In [17]:
DB_CONFIG = {
    'host': '123.30.48.173',
    'port': 5432,
    'dbname': 'metrics_db',
    'user': 'mlops',
    'password': 'mlops123',
}

# Load data

In [18]:
def load_metrics() -> pd.DataFrame:
    conn = psycopg2.connect(**DB_CONFIG)
    df = pd.read_sql(
        "SELECT ts, value FROM metrics WHERE metric_name='cpu_usage_active' ORDER BY ts",
        con=conn,
    )
    conn.close()
    return df

In [19]:
def load_labels(metric_group='cpu') -> pd.DataFrame:
    conn = psycopg2.connect(**DB_CONFIG)
    df = pd.read_sql(
        "SELECT start_ts, end_ts, label " \
        "FROM labeled_events " \
        "WHERE metric_group=%s;",
        con=conn,
        params=[metric_group],
    )
    conn.close()
    return df

# Label data

In [20]:
def attach_labels(metrics_df: pd.DataFrame, labels_df: pd.DataFrame) -> pd.DataFrame:
    metrics_df['true_label'] = 'unlabeled'
    for _, row in labels_df.iterrows():
        mask = (metrics_df["ts"] >= row["start_ts"]) & (metrics_df["ts"] <= row["end_ts"])
        metrics_df.loc[mask, 'true_label'] = row['label']
    return metrics_df

# Build features

In [24]:
def build_features(df: pd.DataFrame, window: int = 4) -> pd.DataFrame:
    df["roll_mean"] = df["value"].rolling(window, min_periods=1).mean()
    df["roll_std"] = df["value"].rolling(window, min_periods=1).std().fillna(0)
    df["diff"] = df["value"].diff().fillna(0)
    return df

# Main code

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("anomaly-detection-cpu")

metrics_df = load_metrics()
labels_df = load_labels('cpu')
metrics_df = attach_labels(metrics_df, labels_df)
metrics_df = build_features(metrics_df)

feature_cols = ['value', 'roll_mean', 'roll_std', 'diff']

labeled_df = metrics_df[metrics_df['true_label'].isin(['normal', 'anomaly'])]

print(f"Có {len(labeled_df)} điểm có nhãn thật ({(labeled_df['true_label']=='anomaly').sum()} anomaly)")

contamination = 0.5
model = IsolationForest(n_estimators=200, contamination=contamination, random_state=42)
model.fit(metrics_df[feature_cols])  # train trên toàn bộ, kể cả chưa gán nhãn

# Đánh giá TRÊN PHẦN CÓ NHÃN THẬT
X_labeled = labeled_df[feature_cols]
y_true = (labeled_df["true_label"] == "anomaly").astype(int)
y_pred = (model.predict(X_labeled) == -1).astype(int)

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

with mlflow.start_run(run_name='cpu_anomaly.v1'):
    mlflow.log_param("contamination", contamination)
    mlflow.log_param("window", 12)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("n_labeled_samples", len(labeled_df))
    mlflow.sklearn.log_model(model, "model", registered_model_name="anomaly-detector-cpu")


C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_8808\264096420.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_8808\405673637.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


Có 198 điểm có nhãn thật (132 anomaly)
Precision=0.729, Recall=0.977, F1=0.835


2026/08/12 17:05:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/12 17:05:41 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\NGUYEN~1\AppData\Local\Temp\tmptg2x4a0e\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 
Registered model 'anomaly-detector-cpu' already exists. Creating a new version of this model...
2026/08/12 17:05:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: anomaly-detector-cpu, version 3


🏃 View run wistful-vole-521 at: http://localhost:5000/#/experiments/1/runs/45406cce78534fe99b6a241b594f3e06
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '3' of model 'anomaly-detector-cpu'.


In [26]:
print(f"Tỷ lệ anomaly thật trong tập labeled: {(labeled_df['true_label']=='anomaly').mean():.2%}")

Tỷ lệ anomaly thật trong tập labeled: 66.67%


In [29]:
# Set contamination gần với tỷ lệ THẬT trong data train (không phải data labeled riêng)
actual_anomaly_rate = (labeled_df['true_label']=='anomaly').mean()
contamination = min(max(actual_anomaly_rate, 0.01), 0.5)  # kẹp trong khoảng hợp lý
# model = IsolationForest(n_estimators=200, contamination=contamination, random_state=42)
actual_anomaly_rate, contamination

(np.float64(0.6666666666666666), 0.5)

# scrap this

In [6]:
import pandas as pd

df = pd.DataFrame({
    "thoi_gian": [1, 2, 3, 4, 5],
    "cpu": [10, 12, 95, 98, 15],
    "nhan": ["", "", "", "", ""]
})

In [8]:
dieu_kien = df["cpu"] > 80
dieu_kien

0    False
1    False
2     True
3     True
4    False
Name: cpu, dtype: bool

In [9]:
df.loc[dieu_kien, "nhan"] = "Qua tai"
df

,thoi_gian,cpu,nhan
0,1,10,
1,2,12,
2,3,95,Qua tai
3,4,98,Qua tai
4,5,15,
